In [10]:
from pyspark.sql import SparkSession

import os
from pathlib import Path

from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np


In [11]:
spark = SparkSession.builder \
        .master("local") \
        .appName("Brain Tumor Classification") \
        .getOrCreate()

In [12]:
def find_project_root():

    current = Path.cwd()

    for _ in range(5):
        if (current / ".git").exists():
            data_path = current / "data" / "raw"
            if data_path.exists():
                return data_path
        current = current.parent

    raise FileNotFoundError(
        "Projet root path could not be found"
    )

data_path = find_project_root()


In [13]:
data_path

WindowsPath('c:/Code/Cours/sparkcore/Brain-Tumor-MRI-Classification/data/raw')

In [14]:
classes = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
print(f"Nombre de classes: {len(classes)}")
print(f"\nClasses :\n{chr(10).join(classes)}")

Nombre de classes: 44

Classes :
Astrocitoma T1
Astrocitoma T1C+
Astrocitoma T2
Carcinoma T1
Carcinoma T1C+
Carcinoma T2
Ependimoma T1
Ependimoma T1C+
Ependimoma T2
Ganglioglioma T1
Ganglioglioma T1C+
Ganglioglioma T2
Germinoma T1
Germinoma T1C+
Germinoma T2
Glioblastoma T1
Glioblastoma T1C+
Glioblastoma T2
Granuloma T1
Granuloma T1C+
Granuloma T2
Meduloblastoma T1
Meduloblastoma T1C+
Meduloblastoma T2
Meningioma T1
Meningioma T1C+
Meningioma T2
Neurocitoma T1
Neurocitoma T1C+
Neurocitoma T2
Oligodendroglioma T1
Oligodendroglioma T1C+
Oligodendroglioma T2
Papiloma T1
Papiloma T1C+
Papiloma T2
Schwannoma T1
Schwannoma T1C+
Schwannoma T2
Tuberculoma T1
Tuberculoma T1C+
Tuberculoma T2
_NORMAL T1
_NORMAL T2


In [15]:
class_counts = {}
total_images = 0

for class_name in classes:
    class_path = data_path / class_name
    count = sum(1 for pattern in ["*.jpeg", "*.JPG", "*.jpg"] for _ in class_path.glob(pattern))
    class_counts[class_name] = count
    total_images += count

df_counts = pd.DataFrame(list(class_counts.items()), columns=["Classe", "Nombre"])
df_counts = df_counts.sort_values("Nombre", ascending=False)

print(f"\nNombre total d'images : {total_images}")
print(f"\nDistribution par classe :\n{df_counts.to_string(index=False)}")


Nombre total d'images : 5744

Distribution par classe :
                Classe  Nombre
       Meningioma T1C+     587
         Meningioma T1     403
         Meningioma T2     367
            _NORMAL T1     322
      Neurocitoma T1C+     299
            _NORMAL T2     289
      Astrocitoma T1C+     279
       Schwannoma T1C+     230
        Astrocitoma T1     224
        Astrocitoma T2     211
         Papiloma T1C+     187
         Schwannoma T1     179
        Neurocitoma T1     169
         Schwannoma T2     156
        Carcinoma T1C+     122
        Neurocitoma T2     118
           Papiloma T1     104
     Glioblastoma T1C+     100
      Tuberculoma T1C+      87
  Oligodendroglioma T1      86
          Carcinoma T2      83
           Papiloma T2      79
         Ependimoma T2      77
   Meduloblastoma T1C+      75
Oligodendroglioma T1C+      72
          Carcinoma T1      72
  Oligodendroglioma T2      66
       Ependimoma T1C+      62
       Glioblastoma T2      60
       Gliobl

In [16]:
print(f"\n{'='*50}")
print(f"STATISTIQUES DE DISTRIBUTION")
print(f"{'='*50}")
print(f"Moyenne     : {df_counts["Nombre"].mean():.2f} images/classe")
print(f"Médiane     : {df_counts["Nombre"].median():.0f} images/classe")
print(f"Minimum     : {df_counts["Nombre"].min()} images")
print(f"Maximum     : {df_counts["Nombre"].max()} images")
print(f"Ecart-type  : {df_counts["Nombre"].std():.2f}")
print(f"{'='*50}\n")


STATISTIQUES DE DISTRIBUTION
Moyenne     : 130.55 images/classe
Médiane     : 78 images/classe
Minimum     : 20 images
Maximum     : 587 images
Ecart-type  : 122.91



In [17]:
dimensions = []

for class_name in classes:
    class_path = data_path / class_name
    for pattern in ["*.jpeg", "*.JPG", "*.jpg"]:
        for img_path in class_path.glob(pattern):
            img = Image.open(img_path)
            dimensions.append(img.size)

df_dims = pd.DataFrame(dimensions, columns=["Largeur", "Hauteur"])
print(f"\nDimensions des images :")
print(df_dims.describe())


Dimensions des images :
           Largeur      Hauteur
count  5744.000000  5744.000000
mean    548.694116   596.370996
std      91.192133    56.091860
min     305.000000   347.000000
25%     469.000000   571.000000
50%     571.000000   630.000000
75%     630.000000   630.000000
max     630.000000   630.000000
